In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path


In [2]:
# =============================================================================
# ── 4. SILVER LAYER: Aggregation & Transformation (Commune level + Unpivot)
# =============================================================================
print("\n⚙️ Processing BRONZE -> SILVER...")

# Load Bronze
path_rhone_bronze = '../../../data/bronze/2022-pres-t2-commune-rhone-69-bronze.csv'

df_b = pd.read_csv(path_rhone_bronze, sep=";", dtype=str)

# --- A. Aggregate the Base Commune Metrics ---
base_cols = ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune']
metrics_cols = ['Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés']

# Convert metrics to integer for summing
for c in metrics_cols:
    df_b[c] = df_b[c].astype(int)




⚙️ Processing BRONZE -> SILVER...


In [3]:
# Group by Commune and sum the metrics
df_commune = df_b.groupby(base_cols)[metrics_cols].sum().reset_index()

# Recalculate percentages correctly at the commune level
df_commune['% Abs/Ins'] = (df_commune['Abstentions'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Vot/Ins'] = (df_commune['Votants'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Ins'] = (df_commune['Blancs'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Blancs'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Nuls/Ins'] = (df_commune['Nuls'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Nuls/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Nuls'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Exp/Ins'] = (df_commune['Exprimés'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Exp/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Exprimés'] / df_commune['Votants'] * 100), 0).round(2)



In [4]:
# --- B. Unpivot / Melt the Candidates Data ---
cand_dfs = []
for i in range(1, 3):
    # Extract columns for this specific candidate
    cand_subset = base_cols + [f'N°Panneau_{i}', f'Sexe_{i}', f'Nom_{i}', f'Prénom_{i}', f'Voix_{i}']
    temp = df_b[cand_subset].copy()
    
    # Standardize column names
    temp.rename(columns={
        f'N°Panneau_{i}': 'N°Panneau',
        f'Sexe_{i}': 'Sexe',
        f'Nom_{i}': 'Nom', 
        f'Prénom_{i}': 'Prénom',
        f'Voix_{i}': 'Voix'
    }, inplace=True)
    cand_dfs.append(temp)



SyntaxError: invalid non-printable character U+00A0 (2442360942.py, line 12)

In [ ]:
# Combine all candidates vertically
df_cands = pd.concat(cand_dfs, ignore_index=True)
df_cands['Voix'] = df_cands['Voix'].astype(int)

# Group by Commune AND Candidate to sum their votes
cand_group_cols = base_cols + ['N°Panneau', 'Sexe', 'Nom', 'Prénom']
df_cands_grouped = df_cands.groupby(cand_group_cols)['Voix'].sum().reset_index()

# --- C. Merge & Finalize Silver Dataset ---
# Join candidate votes with commune total metrics
df_silver = pd.merge(df_cands_grouped, df_commune, on=base_cols, how='left')

# Recalculate Candidate Percentages
df_silver['% Voix/Ins'] = (df_silver['Voix'] / df_silver['Inscrits'] * 100).round(2)
df_silver['% Voix/Exp'] = np.where(df_silver['Exprimés'] > 0, (df_silver['Voix'] / df_silver['Exprimés'] * 100), 0).round(2)



In [ ]:
# Reorder columns for readability
final_cols = base_cols + metrics_cols + [
    '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot',
    '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot',
    'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp'
]
df_silver = df_silver[final_cols]



In [ ]:
# Save to Silver
path_rhone_silver = '../../../data/silver/2022-pres-t2-commune-rhone-69-silver.csv'
df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")

print(f"✅ SILVER dataset created: {path_rhone_silver}")
print(f"📊 Unique Communes: {df_silver['Code de la commune'].nunique()}")
print(f"📊 Total Rows: {len(df_silver)} (Communes × 11 Candidates)")

# Show a clean preview for a single commune (e.g., 'Affoux')
print("\n🔍 Sample preview (Commune: Affoux):")
display(df_silver[df_silver['Libellé de la commune'] == 'Affoux'].head(5))